# Full Source-to-Sample Journey

Stage E notebook. Stitches `slm_plane`, `fourier_filter_plane`, `objective_pupil_plane`, `surface_plane`, and in-medium `sample_plane` propagation. Corrected-interface panels are ideal numerical diagnostics unless hardware correction is explicitly implemented.

## Stage 8.7 Adjustable Quick-Look Guidance

<!-- STAGE87: adjustable quicklook guidance -->

For fast parameter scouting, use `notebooks/quicklook/00_quick_beam_to_sample_simulator.ipynb`. This notebook remains on its locked stage path: existing execution logic, propagation-power labels, material-proxy caveats, and governance routing are unchanged.

Safe local edits are the explicit config variables already exposed by this notebook, or a copied exploratory run. Keep `fail` and `marginal` labels visible. If a displayed image is visually smoothed, treat that as display interpolation only; rerun balanced/publication sampling before numerical interpretation.


In [1]:
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_metrics, vbb_regime, vbb_studies, vbb_style, viz_fields
from vbb_study.publication import lab_realism as lab_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
out_fig = PATHS["figures"] / "stage_e"
out_csv = PATHS["csv"] / "stage_e"
out_fig.mkdir(parents=True, exist_ok=True)
out_csv.mkdir(parents=True, exist_ok=True)
vbb_style.apply_style()

base0 = bt.default_config(PRESET)
general_grid = replace(base0.grid, axial_points=61, coarse_scan_points=25, crop_pixels=160)
limits_grid = replace(base0.grid, axial_points=151, coarse_scan_points=37, crop_pixels=160)
base0 = replace(base0, grid=general_grid)

def _route_method(method):
    return "physical_axicon" if method == "physical" else "holographic_axicon"

def _route_hardware_status(method, variant):
    if variant == "ideal":
        return "simulation_only"
    return "future_hardware_required" if method == "physical" else "current_lab_realizable"


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [2]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides to `base0` before running any study cell below.
from vbb_study.publication import notebook_controls as nb_controls
from vbb_study.config import um as _um

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='lab_realism',
    # ── edit these to override the base configuration ───────────────────────
    ell=3,
    target_core_diameter_um=3.0,
    target_bessel_length_um=150.0,
    objective_NA=0.45,
    blaze_period_px=20,
)

# Wire control parameters into `base0` so downstream cells use them.
_p = NOTEBOOK_CONTROLS.parameters or {}
if "ell" in _p:
    base0 = replace(base0, target=replace(base0.target, ell=int(_p["ell"])))
if "target_core_diameter_um" in _p:
    base0 = replace(base0, target=replace(base0.target, target_core_diameter_m=float(_p["target_core_diameter_um"]) * _um))
if "target_bessel_length_um" in _p:
    base0 = replace(base0, target=replace(base0.target, target_bessel_length_m=float(_p["target_bessel_length_um"]) * _um))
if "objective_NA" in _p:
    base0 = replace(base0, objective=replace(base0.objective, NA=float(_p["objective_NA"])))
if "blaze_period_px" in _p:
    base0 = replace(base0, slm=replace(base0.slm, blaze_period_px=int(_p["blaze_period_px"])))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


,control,value
0,stage,lab_realism
1,run_mode,balanced
2,save_outputs,False
3,use_canonical_outputs,True
4,allow_publication_export,False
5,notes,Edit for exploration; keep QA labels/caveats v...
6,ell,3
7,target_core_diameter_um,3.0
8,target_bessel_length_um,150.0
9,objective_NA,0.45


In [3]:
# Interactive quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved.
from vbb_study.publication import notebook_widgets as nbw

_panel = nbw.interactive_quicklook(base0, method='holographic', preset='fast')
display(_panel)


In [4]:
def configured_case(method, regime, variant):
    grid = limits_grid if regime == 'limits' else general_grid
    cfg = replace(base0, generation_method=method, grid=grid)
    cfg = vbb_regime.config_for_regime(cfg, regime)
    if method == 'physical':
        levels = 256 if variant == 'lab' else None
        cfg = replace(cfg, physical_axicon=replace(cfg.physical_axicon, slm2_stroke_levels=levels, slm2_conjugate_mode='preserve_vortex'))
        path = 'ideal'
    else:
        path = 'realistic' if variant == 'lab' else 'ideal'
    return cfg, path

def panel_validity(cfg, journey):
    design = bt.compute_design_from_targets(cfg.laser, cfg.target, cfg.material)
    sampling = vbb_regime.sampling_validity(cfg, design, result={'volume': journey.sample_result.volume_result})
    qa = bt.sampling_report(cfg, design, {'volume': journey.sample_result.volume_result})
    order = journey.air_result.get('order', {})
    first_fraction = float(journey.air_result.get('first_order_selected_fraction', np.nan))
    reasons = []
    if not bool(sampling.get('valid', True)):
        reasons.extend(str(v) for v in sampling.get('violations', []))
    if str(qa.get('phase_sampling_label', 'pass')) != 'pass':
        reasons.append(f"phase_sampling_{qa.get('phase_sampling_label')}")
    if str(qa.get('axial_sampling_label', 'pass')) != 'pass':
        reasons.append(f"axial_sampling_{qa.get('axial_sampling_label')}")
    if order and not bool(order.get('first_order_geometry_valid', True)):
        reasons.append('first_order_geometry_invalid')
    if np.isfinite(first_fraction) and first_fraction < 0.05:
        reasons.append(f'low_first_order_fraction={first_fraction:.2e}')
    valid = len(reasons) == 0
    return {
        'valid': bool(valid),
        'reasons': reasons,
        'sampling': sampling,
        'qa': qa,
        'first_order_selected_fraction': first_fraction,
    }

def journey_winding_fields(journey, method, cfg):
    air = journey.air_result
    surface = air["surface_field"]
    design = air["design"]
    measured = viz_fields.phase_winding(
        surface.Ex, surface.grid, float(design.vortex_main_ring_radius_m)
    )
    error = abs(measured - int(design.ell))
    if method == "physical" and error >= 0.1:
        raise RuntimeError(
            f"physical winding {measured:.6g} does not match requested ell={design.ell}"
        )
    return {
        "requested_vortex_charge": int(design.ell),
        "measured_winding": float(measured),
        "winding_error": float(error),
        "winding_pass": bool(error < 0.1),
        "slm2_conjugate_mode": (
            str(cfg.physical_axicon.slm2_conjugate_mode) if method == "physical" else "not_applicable"
        ),
        "vortex_removal_acknowledged": (
            bool(cfg.physical_axicon.allow_vortex_removal) if method == "physical" else False
        ),
        "propagation_power_drift_fraction": air["volume"].get("propagation_power_drift_fraction"),
        "propagation_power_label": air["volume"].get("propagation_power_label"),
        "quantitative_metrics_valid": air["volume"].get("quantitative_metrics_valid"),
        "quantitative_metrics_invalid_reason": air["volume"].get("quantitative_metrics_invalid_reason"),
    }

def journey_row(label, regime, method, variant, correction, journey, validity, winding):
    m = journey.metrics
    sm = journey.sample_result.metrics
    return {
        'label': label, 'regime': regime, 'method': method, 'variant': variant, 'correction': correction, 'interface_correction_label': correction, 'interface_correction_label': correction, 'interface_correction_label': correction,
        'validity_valid': validity['valid'],
        'validity_reasons': ';'.join(validity['reasons']),
        'first_order_selected_fraction': validity['first_order_selected_fraction'],
        'continuity_relative_power_error': m['continuity_relative_power_error'],
        'target_write_depth_um': m['target_write_depth_um'],
        'sample_zone_center_um': m['sample_zone_center_um'],
        'sample_zone_center_error_um': m['sample_zone_center_error_um'],
        'sample_bessel_zone_um': m['sample_bessel_zone_um'],
        'full_axial_dz_um': m['full_axial_dz_um'],
        'air_axial_dz_um': m['air_axial_dz_um'],
        'sample_axial_dz_um': m['sample_axial_dz_um'],
        'corrected_rel_l2_to_no_interface': sm['corrected_rel_l2_to_no_interface'],
        'uncorrected_rel_l2_to_no_interface': sm['uncorrected_rel_l2_to_no_interface'],
        'phase_only_power_relative_error': sm['phase_only_power_relative_error'],
        'interface_spherical_after_waves': sm['interface_spherical_after_waves'],
        **winding,
    }


In [5]:
def _shared_linear_display(values, vmax):
    arr = np.maximum(np.asarray(values, dtype=float), 0.0)
    normalised = np.clip(arr / (float(vmax) + bt.EPS), 0.0, 1.0)
    return vbb_style.display_scale(normalised, gamma=0.45)

def _is_first_order_impossible(validity):
    return any('first_order_geometry_invalid' in r for r in validity['reasons'])

def _stamp_invalid(ax, validity):
    if validity['valid']:
        return
    reasons = [r for r in validity['reasons'][:3] if 'first_order_geometry_invalid' not in r and 'low_first_order' not in r]
    if not reasons:
        return
    text = 'OUT OF VALIDITY\n' + '\n'.join(reasons)
    ax.text(
        0.5, 0.5, text,
        transform=ax.transAxes,
        ha='center', va='center',
        color='white', fontsize=9, fontweight='bold',
        bbox={'boxstyle': 'round,pad=0.35', 'facecolor': '#D55E00', 'edgecolor': 'white', 'alpha': 0.88},
        zorder=20,
    )
    ax.text(
        0.02, 0.98, 'diagnostic only',
        transform=ax.transAxes,
        ha='left', va='top',
        color='white', fontsize=8,
        bbox={'boxstyle': 'round,pad=0.20', 'facecolor': 'black', 'edgecolor': 'none', 'alpha': 0.65},
        zorder=20,
    )

def plot_journey_grid(regime, method, journeys, validities, output_path, *, charge_label=None):
    fig, axes = plt.subplots(3, 3, figsize=(13.5, 9.5), constrained_layout=True)

    # Compute shared vmax only from panels where first-order isolation is not fundamentally broken
    valid_peaks = [
        float(np.nanpercentile(np.asarray(j.volume['xz'], dtype=float), 99.5))
        for key, j in journeys.items() if not _is_first_order_impossible(validities[key])
    ]
    if not valid_peaks:
        valid_peaks = [float(np.nanpercentile(np.asarray(j.volume['xz'], dtype=float), 99.5)) for j in journeys.values()]
    shared_vmax = max(max(valid_peaks), bt.EPS)

    image_artist = None
    for r, variant in enumerate(('ideal', 'lab')):
        for c, corr in enumerate(('uncorrected_interface', 'ideal_numerical_correction')):
            key = (variant, corr)
            j = journeys[key]
            validity = validities[key]
            ax = axes[r, c]
            ax.set_title(f'{variant} | {corr.replace("_", " ")}')
            ax.set_xlabel('z [um from surface; air <0, sample >0]')
            ax.set_ylabel('x [um, focused/sample plane]')

            if _is_first_order_impossible(validity):
                # Cone ring frequency exceeds carrier: first-order isolation is physically impossible.
                # Showing the raw field (essentially noise) would be misleading, so show a blank panel.
                ax.set_facecolor('#0a0a0a')
                ax.text(0.5, 0.60, 'NOT ACHIEVABLE', transform=ax.transAxes,
                        ha='center', va='center', color='#E07040', fontsize=11, fontweight='bold', zorder=10)
                _ord = j.air_result.get('order', {})
                carrier = float(_ord.get('carrier_lpmm', float('nan')))
                cone = float(_ord.get('axicon_cone_radius_lpmm', float('nan')))
                frac = validity.get('first_order_selected_fraction', float('nan'))
                carrier_str = f'{carrier:.2f}' if np.isfinite(carrier) else '?'
                cone_str = f'{cone:.2f}' if np.isfinite(cone) else '?'
                ax.text(0.5, 0.42,
                        f'carrier {carrier_str} lp/mm  <  cone {cone_str} lp/mm\n'
                        f'(first-order isolation impossible; {frac:.1e} power in order)',
                        transform=ax.transAxes, ha='center', va='center',
                        color='white', fontsize=8, zorder=10)
                ax.text(0.02, 0.98, 'not shown — physically impossible', transform=ax.transAxes,
                        ha='left', va='top', color='#E07040', fontsize=8, zorder=10)
                continue

            vol = j.volume
            z_um = np.asarray(vol['z'], dtype=float) / bt.um
            x_um = np.asarray(vol['crop_grid']['x'], dtype=float) / bt.um
            image_artist = ax.imshow(
                _shared_linear_display(vol['xz'], shared_vmax),
                origin='lower', aspect='auto',
                extent=[float(z_um[0]), float(z_um[-1]), float(x_um[0]), float(x_um[-1])],
                cmap=vbb_style.INTENSITY_CMAP, vmin=0.0, vmax=1.0,
            )
            ax.axvspan(float(z_um[0]), 0.0, color='#56B4E9', alpha=0.08, lw=0)
            ax.axvspan(0.0, float(z_um[-1]), color='#009E73', alpha=0.07, lw=0)
            ax.axvline(0.0, color='white', lw=1.0, ls='-', alpha=0.85)
            ax.axvline(j.metrics['target_write_depth_um'], color='cyan', lw=1.0, ls='--', alpha=0.95)
            per_z = vbb_metrics.per_z_metrics_from_volume(
                vol, ell=int(j.sample_result.metrics['ell']), kr_m_inv=float(j.sample_result.metrics['kr_sample_m_inv']),
                center_mode='centroid', reference_index=int(vol.get('peak_index', 0))
            )
            radius_key = 'core_radius_m' if int(j.sample_result.metrics['ell']) == 0 else 'ring_radius_m'
            radius_um = np.asarray(per_z[radius_key], dtype=float) / bt.um
            ax.plot(z_um, radius_um, color='white', lw=0.8, alpha=0.85)
            ax.plot(z_um, -radius_um, color='white', lw=0.8, alpha=0.85)
            _stamp_invalid(ax, validity)

    axes[0, 2].axis('off')
    axes[0, 2].text(
        0.0, 0.98,
        'Blue shading: air (n=1)\nGreen shading: Cr:ZnSe (n=2.44)\nWhite line: surface\nCyan line: target write depth',
        va='top', ha='left', fontsize=10,
    )
    lab_corr = journeys[('lab', 'ideal_numerical_correction')]
    lab_valid = validities[('lab', 'ideal_numerical_correction')]
    _rows = [
        ('validity', 'PASS' if lab_valid['valid'] else 'OUT OF VALIDITY'),
        ('continuity err', f"{lab_corr.metrics['continuity_relative_power_error']:.2e}"),
        ('target depth', f"{lab_corr.metrics['target_write_depth_um']:.1f} um"),
        ('corrected zone centre', f"{lab_corr.metrics['sample_zone_center_um']:.1f} um"),
        ('zone-centre error', f"{lab_corr.metrics['sample_zone_center_error_um']:.1f} um"),
        ('sample dz', f"{lab_corr.metrics['sample_axial_dz_um']:.2f} um"),
        ('shared scale vmax', f"{shared_vmax:.2e}"),
    ]
    axes[1, 2].axis('off')
    table = axes[1, 2].table(cellText=_rows, colLabels=['check', 'value'], loc='center')
    table.auto_set_font_size(False); table.set_fontsize(8.5); table.scale(1.0, 1.18)

    # Axial traces: normalise to the ideal-corrected peak so both series share one absolute scale.
    # This prevents noise (from a geometry-invalid lab run) from being rescaled to look like a beam.
    for c, corr in enumerate(('uncorrected_interface', 'ideal_numerical_correction')):
        ax = axes[2, c]
        ideal_peak = np.asarray(journeys[('ideal', corr)].volume['peak'], dtype=float)
        ref_max = float(np.nanmax(ideal_peak)) + bt.EPS
        for variant, ls in (('ideal', '-'), ('lab', '--')):
            j = journeys[(variant, corr)]
            validity = validities[(variant, corr)]
            z_um = np.asarray(j.volume['z'], dtype=float) / bt.um
            if _is_first_order_impossible(validity):
                frac = validity.get('first_order_selected_fraction', float('nan'))
                ax.annotate(
                    f'lab: first-order isolation impossible (cone > carrier)\n'
                    f'  {frac:.1e} of power in order — trace omitted',
                    xy=(0.02, 0.10), xycoords='axes fraction',
                    ha='left', va='bottom', color='#D55E00', fontsize=8,
                )
                continue
            peak = np.asarray(j.volume['peak'], dtype=float)
            ax.plot(z_um, peak / ref_max, ls=ls, label=f'{variant} peak')
        target = journeys[('lab', corr)].metrics['target_write_depth_um']
        ax.axvline(0.0, color='0.2', lw=1.0, label='surface' if c == 0 else None)
        ax.axvline(target, color='#0072B2', lw=1.0, ls='--', label='target depth' if c == 0 else None)
        ax.set_title(f'axial trace | {corr}')
        ax.set_xlabel('z [um from surface; air <0, sample >0]')
        ax.set_ylabel('normalised peak intensity\n(shared ideal-corrected scale)')
        ax.legend(loc='best')

    axes[2, 2].axis('off')
    reason_text = '\n'.join(lab_valid['reasons']) if lab_valid['reasons'] else 'none'
    axes[2, 2].text(
        0.0, 0.98,
        'Heatmaps: shared gamma-corrected (γ=0.45) scale per figure.\n'
        'Axial trace normalised to ideal-corrected peak max.\n\nLab corrected validity reasons:\n' + reason_text,
        va='top', ha='left', fontsize=9,
    )
    if image_artist is not None:
        cbar = fig.colorbar(image_artist, ax=axes[:2, :2], shrink=0.88)
        cbar.set_label('shared gamma-corrected (γ=0.45) XZ intensity [a.u.; 99.5th pct]')
    suptitle = f'Full journey: {regime} {method}'
    if charge_label:
        suptitle += f'\n{charge_label}'
    fig.suptitle(suptitle)
    caption = (
        f'Full source-to-sample journey for {regime} {method}. The z-axis is continuous with the surface at 0 um; '
        'air is z<0 and Cr:ZnSe is z>0. Heatmaps share one gamma-corrected (γ=0.45) intensity scale across valid panels; '
        'out-of-validity panels are stamped. Panels where first-order isolation is physically impossible '
        '(cone ring frequency > SLM carrier) are blanked rather than showing misleading noise.'
    )
    if charge_label:
        caption += f' {charge_label}.'
    out = vbb_style.save_figure(fig, output_path, caption, metadata={'study_kind': 'full_source_to_sample', 'regime': regime, 'method': method})
    plt.close(fig)
    return out

In [6]:
from vbb_study.viz_fields import measured_charge_label as _mcl_nb06
rows = []
all_results = {}
all_validities = {}
for regime in ('general', 'limits'):
    for method in ('holographic', 'physical'):
        block = {}
        validity_block = {}
        for variant in ('ideal', 'lab'):
            cfg, path = configured_case(method, regime, variant)
            for correction, correct in (('uncorrected_interface', False), ('ideal_numerical_correction', True)):
                label = f'{regime}_{method}_{variant}_{correction}'
                journey = vbb_studies.run_full_source_to_sample(cfg, correct_interface=correct, path=path)
                validity = panel_validity(cfg, journey)
                winding = journey_winding_fields(journey, method, cfg)
                rows.append(journey_row(label, regime, method, variant, correction, journey, validity, winding))
                block[(variant, correction)] = journey
                validity_block[(variant, correction)] = validity
                all_results[label] = journey
                all_validities[label] = validity
        # Compute charge label from lab uncorrected air result if surface_field is accessible
        _charge_lbl = None
        _lab_j = block.get(('lab', 'uncorrected_interface'))
        if _lab_j is not None:
            _air = getattr(_lab_j, 'air_result', {}) or {}
            _sf_nb06 = _air.get('surface_field') if hasattr(_air, 'get') else None
            _des_nb06 = _air.get('design') if hasattr(_air, 'get') else None
            if _sf_nb06 is not None and _des_nb06 is not None:
                _charge_lbl = _mcl_nb06(
                    _sf_nb06.Ex, _sf_nb06.grid,
                    float(_des_nb06.vortex_main_ring_radius_m),
                    design_ell=int(_des_nb06.ell),
                    conjugate_mode='preserve_vortex' if method == 'physical' else None,
                )
        plot_journey_grid(
            regime, method, block, validity_block,
            out_fig / f'full_source_to_sample_journey_{regime}_{method}.png',
            charge_label=_charge_lbl,
        )
raw_summary = pd.DataFrame(rows)
stamped_rows = []
for row in raw_summary.to_dict("records"):
    label = row["interface_correction_label"]
    method = row["method"]
    variant = row["variant"]
    lab_schema.annotate_lab_realism_row(
        row,
        generation_method="interface_corrected_numerical" if label == "ideal_numerical_correction" else "interface_uncorrected",
        model_level="interface_model",
        hardware_status="diagnostic_only" if label == "ideal_numerical_correction" else _route_hardware_status(method, variant),
        plane_label="propagation_axis_z",
        coordinate_frame="stitched_air_surface_and_in_medium_z_um",
        run_id=RUN_ID,
        preset=PRESET,
        path="full_source_to_sample",
    )
    row["route_generation_method"] = _route_method(method)
    stamped_rows.append(row)
summary = lab_schema.ordered_lab_realism_frame(stamped_rows)
summary.to_csv(out_csv / "full_source_to_sample_journey_summary.csv", index=False)
summary

,run_id,generated_at_utc,source_schema_version,preset,path,generation_method,model_level,hardware_status,plane_label,coordinate_frame,...,sample_zone_center_error_um,sample_bessel_zone_um,full_axial_dz_um,air_axial_dz_um,sample_axial_dz_um,corrected_rel_l2_to_no_interface,uncorrected_rel_l2_to_no_interface,phase_only_power_relative_error,interface_spherical_after_waves,route_generation_method
0,20260715T180543Z,2026-07-15T18:05:43.463219+00:00,1.0.0,fast,full_source_to_sample,interface_uncorrected,interface_model,simulation_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,126.237298,122.525404,8.125,6.250000,8.125,0.007046,1.350925,0.000000e+00,-0.000089,holographic_axicon
1,20260715T180543Z,2026-07-15T18:05:43.463243+00:00,1.0.0,fast,full_source_to_sample,interface_corrected_numerical,interface_model,diagnostic_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,-7.408189,292.300386,8.125,6.250000,8.125,0.007046,1.350925,0.000000e+00,-0.000089,holographic_axicon
2,20260715T180543Z,2026-07-15T18:05:43.463254+00:00,1.0.0,fast,full_source_to_sample,interface_uncorrected,interface_model,current_lab_realizable,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,125.169362,124.661275,8.125,6.250000,8.125,0.007087,1.357514,3.185409e-16,-0.000161,holographic_axicon
3,20260715T180543Z,2026-07-15T18:05:43.463262+00:00,1.0.0,fast,full_source_to_sample,interface_corrected_numerical,interface_model,diagnostic_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,-20.174831,71.794810,8.125,6.250000,8.125,0.007087,1.357514,3.185409e-16,-0.000161,holographic_axicon
4,20260715T180543Z,2026-07-15T18:05:43.463268+00:00,1.0.0,fast,full_source_to_sample,interface_uncorrected,interface_model,simulation_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,131.939521,111.120958,8.125,6.250000,8.125,0.007038,1.355268,1.865580e-16,-0.000089,physical_axicon
5,20260715T180543Z,2026-07-15T18:05:43.463275+00:00,1.0.0,fast,full_source_to_sample,interface_corrected_numerical,interface_model,diagnostic_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,-8.789475,294.742994,8.125,6.250000,8.125,0.007038,1.355268,1.865580e-16,-0.000089,physical_axicon
6,20260715T180543Z,2026-07-15T18:05:43.463281+00:00,1.0.0,fast,full_source_to_sample,interface_uncorrected,interface_model,future_hardware_required,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,131.996519,111.006962,8.125,6.250000,8.125,0.007038,1.355321,5.596993e-16,-0.000089,physical_axicon
7,20260715T180543Z,2026-07-15T18:05:43.463288+00:00,1.0.0,fast,full_source_to_sample,interface_corrected_numerical,interface_model,diagnostic_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,-8.766873,294.773856,8.125,6.250000,8.125,0.007038,1.355321,5.596993e-16,-0.000089,physical_axicon
8,20260715T180543Z,2026-07-15T18:05:43.463294+00:00,1.0.0,fast,full_source_to_sample,interface_uncorrected,interface_model,simulation_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,-257.406835,85.186330,4.500,5.000000,4.500,0.007180,1.399812,1.516395e-16,-0.000089,holographic_axicon
9,20260715T180543Z,2026-07-15T18:05:43.463300+00:00,1.0.0,fast,full_source_to_sample,interface_corrected_numerical,interface_model,diagnostic_only,propagation_axis_z,stitched_air_surface_and_in_medium_z_um,...,-20.340013,274.125881,4.500,5.000000,4.500,0.007180,1.399812,1.516395e-16,-0.000089,holographic_axicon


In [7]:
# Holographic phase hero panels: domain-coloured phase showing ℓ=3 winding is preserved.
# Uses air-result surface_field from the lab corrected journey if accessible.
# Task G: preset/N/device_downsample stated in caption.
import matplotlib.pyplot as plt
from vbb_study.viz_fields import complex_field_image, measured_charge_label as _mcl_hero

_N_nb06 = base0.grid.N
_ds_nb06 = base0.grid.device_downsample
for regime in ('general', 'limits'):
    lbl_key = f'{regime}_holographic_lab_ideal_numerical_correction'
    if lbl_key not in all_results:
        print(f"  {regime}: journey not in all_results, skipping hero panel")
        continue
    journey = all_results[lbl_key]
    _air = getattr(journey, 'air_result', {}) or {}
    _sf_hero = _air.get('surface_field') if hasattr(_air, 'get') else None
    _des_hero = _air.get('design') if hasattr(_air, 'get') else None
    if _sf_hero is None or _des_hero is None:
        print(f"  {regime}: air surface_field not accessible in air_result — skipping hero panel")
        continue
    sample_r_hero = float(_des_hero.vortex_main_ring_radius_m)
    charge_lbl_hero = _mcl_hero(
        _sf_hero.Ex, _sf_hero.grid, sample_r_hero,
        design_ell=int(_des_hero.ell),
    )
    hero_title = (
        f"Holographic {regime} surface-plane phase | {charge_lbl_hero}\n"
        f"[preset={PRESET}, N={_N_nb06}, device_downsample={_ds_nb06}]"
    )
    fig_hero, _ = complex_field_image(
        _sf_hero.Ex, _sf_hero.grid, title=hero_title,
    )
    fig_hero.savefig(
        out_fig / f'nb06_holographic_{regime}_phase_hero.png',
        dpi=150, bbox_inches='tight',
    )
    plt.close(fig_hero)
    print(f"  {regime} holographic hero: {charge_lbl_hero}")

  general holographic hero: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)
  limits holographic hero: measured winding = -1.00 (design ℓ=3; charge stripped)
